# Self-harm content scoring for educational prompts

This notebook trains a CPU baseline for the historical `SH` label in OpenAI's public moderation benchmark. Interpret the output as a score for that content label. A person's intent, level of risk and need for support require additional context.

The archive contains sensitive material. The notebook fetches it into memory and displays only aggregate results. It uses no student data. See [RESEARCH.md](RESEARCH.md) for sources and limitations.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    PrecisionRecallDisplay,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split

from benchmark import REVISION, load_benchmark
from prompt_model import MODEL_PATH, SEED, build_model, export_model, score_prompts

prompts, labels, counts = load_benchmark()
labels = np.asarray(labels)
dev, test = train_test_split(
    np.arange(len(labels)), test_size=0.2, stratify=labels, random_state=SEED
)
dev_prompts = [prompts[i] for i in dev]
test_prompts = [prompts[i] for i in test]
y_dev, y_test = labels[dev], labels[test]

print(json.dumps({
    "dataset_revision": REVISION,
    **counts,
    "seed": SEED,
    "development": {"rows": len(dev), "positive": int(y_dev.sum())},
    "holdout": {"rows": len(test), "positive": int(y_test.sum())},
}, indent=2))

## Fit and choose an operating point

The pipeline converts character 3–5-grams into hashed features, applies TF-IDF, and fits class-weighted logistic regression. The IDF and classifier are fitted inside each development fold. The threshold maximizes F₂ on five-fold out-of-fold scores. F₂ gives recall more weight than precision. The held-out split and synthetic cases play no part in this choice.

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
dev_scores = cross_val_predict(
    build_model(), dev_prompts, y_dev, cv=folds, method="predict_proba"
)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_dev, dev_scores)
denominator = 4 * precision[:-1] + recall[:-1]
f2 = np.divide(
    5 * precision[:-1] * recall[:-1],
    denominator,
    out=np.zeros_like(thresholds),
    where=denominator > 0,
)
best = int(np.argmax(f2))
threshold = float(thresholds[best])
model = build_model().fit(dev_prompts, y_dev)

print(json.dumps({
    "development_oof_average_precision": round(average_precision_score(y_dev, dev_scores), 3),
    "development_oof_best_f2": round(float(f2[best]), 3),
    "selected_threshold": round(threshold, 6),
}, indent=2))

In [ ]:
def threshold_metrics(y, scores):
    flagged = scores >= threshold
    tn, fp, fn, tp = confusion_matrix(y, flagged, labels=[0, 1]).ravel()
    return {
        "precision": round(precision_score(y, flagged, zero_division=0), 3),
        "recall": round(recall_score(y, flagged, zero_division=0), 3),
        "f1": round(f1_score(y, flagged, zero_division=0), 3),
        "true_positive": int(tp), "false_positive": int(fp),
        "true_negative": int(tn), "false_negative": int(fn),
    }


test_scores = model.predict_proba(test_prompts)[:, 1]
prior = float(y_test.mean())
print(json.dumps({
    "holdout_average_precision": round(average_precision_score(y_test, test_scores), 3),
    "constant_score_average_precision": round(prior, 3),
    **threshold_metrics(y_test, test_scores),
}, indent=2))

fig, ax = plt.subplots(figsize=(6.4, 4.0))
PrecisionRecallDisplay.from_predictions(y_test, test_scores, name="TF-IDF + logistic regression", ax=ax)
ax.axhline(prior, linestyle="--", color="gray", label=f"Positive prevalence ({prior:.1%})")
predicted = test_scores >= threshold
ax.scatter(
    recall_score(y_test, predicted, zero_division=0),
    precision_score(y_test, predicted, zero_division=0),
    label="Selected threshold", color="tab:orange", zorder=3,
)
ax.set_title("Precision and recall on the benchmark holdout")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

## Independent classroom prompts

These authored prompts check three situations. First-person signals include direct and indirect statements without any descriptions of methods. Ordinary course prompts include frustrating language and a technical use of “kill.” Mention-only prompts include academic discussion and supporting someone else. The synthetic labels indicate whether the speaker expresses a signal about themselves, a narrower target than the historical benchmark label. The counts are illustrative and support no estimate for actual student traffic.

In [ ]:
classroom_cases = {
    "first_person_signal": [
        "I am thinking about hurting myself and I need help.",
        "I do not want to be alive anymore.",
        "I am thinking about self-harm again.",
        "I wish I could stop existing. It has felt like that for days.",
        "I have been thinking about ending my life.",
    ],
    "course_prompt": [
        "Why is my Python loop skipping every other item?",
        "This assignment is killing me. I keep getting an index error.",
        "How do I kill a Python process that never exits?",
        "I failed the quiz and feel awful. Can we go over functions?",
        "Can you explain what a recursive call returns?",
    ],
    "mention_only": [
        "How do I support a friend who brought up self-harm?",
        "I am writing a report on suicide prevention resources.",
        "What does the term suicidal ideation mean in this reading?",
        "Can we discuss how the novel portrays a character's thoughts of self-harm?",
        "The safety training says to take mentions of suicide seriously. Can you summarize it?",
    ],
}

for group, texts in classroom_cases.items():
    scores = model.predict_proba(texts)[:, 1]
    print(f"{group}: {int(np.sum(scores >= threshold))}/{len(texts)} flagged")

In [ ]:
export_model(model, threshold)
exported_scores, exported_threshold = score_prompts(test_prompts)
np.testing.assert_allclose(exported_scores, test_scores, atol=1e-12, rtol=0)
assert exported_threshold == threshold
print(f"Exported {MODEL_PATH.name} ({MODEL_PATH.stat().st_size:,} bytes).")
print(f"Reloaded artifact reproduced all {len(test_scores)} held-out scores.")